# Synthetic data

A small generator for practice rows. The course version switches OpenAI and Ollama. Here the switch is three Azure deployments: `gpt-4o`, `gpt-4o-mini`, and `gpt-4.1`. Two prompts: a plain request, or the same request with one example row.


## Client

In [1]:
import json
import os
from pathlib import Path

import gradio as gr
from dotenv import load_dotenv
from openai import AzureOpenAI

cwd = Path.cwd().resolve()
repo_root = next((p for p in [cwd, *cwd.parents] if (p / ".git").exists()), cwd)
load_dotenv(repo_root / ".env", override=True)

client = AzureOpenAI(
    api_key=os.getenv("GPT_KEY"),
    api_version=os.getenv("GPT_API_VERSION"),
    azure_endpoint=os.getenv("GPT_ENDPOINT"),
)

deployments = ["gpt-4o", "gpt-4o-mini", "gpt-4.1"]


## Prompts

In [2]:
def messages_for(topic, count, style):
    system = "You write synthetic data. Reply with a JSON list only. No markdown."
    fields = 'Each object has "name", "city", and "note".'
    if style == "With an example":
        user = (
            "Example:\n"
            '[{"name": "Nimal Perera", "city": "Colombo", '
            '"note": "Asked about weekend classes."}]\n\n'
            f"Write {int(count)} new rows about {topic}. {fields}"
        )
    else:
        user = f"Write {int(count)} rows about {topic}. {fields}"
    return [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
    ]


def read_json(text):
    text = text.strip()
    if text.startswith("```"):
        text = text.split("\n", 1)[-1].rsplit("```", 1)[0]
    return json.loads(text)


def generate(topic, count, style, deployment):
    response = client.chat.completions.create(
        model=deployment,
        messages=messages_for(topic, count, style),
        timeout=120.0,
    )
    return read_json(response.choices[0].message.content)


## Page

In [3]:
topic = gr.Textbox("people asking about a Python course", label="Topic")
count = gr.Slider(1, 8, value=3, step=1, label="Rows")
style = gr.Dropdown(["Plain", "With an example"], value="Plain", label="Prompt")
deployment = gr.Dropdown(deployments, value="gpt-4o-mini", label="Model")
table = gr.Dataframe(label="Rows")

gr.Interface(
    fn=generate,
    inputs=[topic, count, style, deployment],
    outputs=table,
    title="Synthetic rows",
    flagging_mode="never",
).launch()


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
